In [ ]:
from pathlib import Path
import json, random, sys, os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from nltk.translate.bleu_score import sentence_bleu

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.wajib.rnn.RNN import RNNScratch
from src.wajib.lstm.LSTM import LSTMScratch
from src.wajib.shared.layers import EmbeddingLayer, DenseLayer
from src.wajib.shared.preprocessing import (
    loadFlickr8kCaptions, loadVocabulary,
 )
from src.wajib.shared.decoder import greedyDecode

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Konfigurasi
FEATURES_NPY  = PROJECT_ROOT / "src/wajib/weights/features/flickr8k_features.npy"
FEATURES_IDX  = PROJECT_ROOT / "src/wajib/weights/features/flickr8k_index.json"
CAPTIONS_FILE = PROJECT_ROOT / "data/flickr8k/captions.txt"
VOCAB_PATH    = PROJECT_ROOT / "src/wajib/weights/vocab.json"
DATA_DIR      = PROJECT_ROOT / "data/flickr8k"
FLICKR_DIR    = DATA_DIR / "Images"

RNN_WEIGHTS_DIR  = PROJECT_ROOT / "src/wajib/weights/rnn"
LSTM_WEIGHTS_DIR = PROJECT_ROOT / "src/wajib/weights/lstm"

EMBED_DIM    = 256
MAX_LEN      = 30
CNN_FEAT_DIM = 2048

features_matrix = np.load(FEATURES_NPY)
with open(FEATURES_IDX) as f:
    idxNames = json.load(f)

imageFeatures = {name: features_matrix[i] for i, name in enumerate(idxNames)}

captionsDict = loadFlickr8kCaptions(str(CAPTIONS_FILE))
vocab = loadVocabulary(str(VOCAB_PATH))
id2word = {v: k for k, v in vocab.items()}

allImages = list(captionsDict.keys())
random.seed(SEED)
random.shuffle(allImages)

trainImgs = set(allImages[:6000])
valImgs   = set(allImages[6000:7000])
testImgs  = set(allImages[7000:])

testCaps = {k: v for k, v in captionsDict.items() if k in testImgs}

print(f"Fitur     : {len(imageFeatures)} gambar")
print(f"Kosa kata : {len(vocab)} token")
print(f"Data uji  : {len(testCaps)} gambar")

In [ ]:
rnnModels = sorted(LSTM_WEIGHTS_DIR.glob("rnn_*.keras"))
lstmModels = sorted(LSTM_WEIGHTS_DIR.glob("lstm_*.keras"))

bestRnnName = "rnn_3L_512h"
bestLstmName = "lstm_3L_512h"

bestRnnPath = RNN_WEIGHTS_DIR / f"{bestRnnName}.keras"
bestLstmPath = LSTM_WEIGHTS_DIR / f"{bestLstmName}.keras"

if not bestRnnPath.exists() or not bestLstmPath.exists():
    if rnnModels:
        bestRnnPath = rnnModels[-1]
        bestRnnName = bestRnnPath.stem
    if lstmModels:
        bestLstmPath = lstmModels[-1]
        bestLstmName = bestLstmPath.stem

print(f"Menggunakan RNN : {bestRnnName}")
print(f"Menggunakan LSTM: {bestLstmName}")

def loadRnnModel(modelPath, numLayers):
    model = tf.keras.models.load_model(str(modelPath))
    
    embed = EmbeddingLayer()
    embed.loadWeights(model.get_layer('embedding'))
    
    proj = DenseLayer()
    proj.loadWeights(model.get_layer('cnn_proj'))
    
    out = DenseLayer(activation='softmax')
    out.loadWeights(model.get_layer('output'))
    
    rnn = RNNScratch()
    rnn.loadWeights([model.get_layer(f'rnn_{i}') for i in range(numLayers)])
    
    return embed, proj, out, rnn

def loadLstmModel(modelPath, numLayers):
    model = tf.keras.models.load_model(str(modelPath))
    
    embed = EmbeddingLayer()
    embed.loadWeights(model.get_layer('embedding'))
    
    proj = DenseLayer()
    proj.loadWeights(model.get_layer('cnn_proj'))
    
    out = DenseLayer(activation='softmax')
    out.loadWeights(model.get_layer('output'))
    
    lstm = LSTMScratch()
    lstm.loadWeights([model.get_layer(f'lstm_{i}') for i in range(numLayers)])
    
    return embed, proj, out, lstm

rnnNumLayers = int(bestRnnName.split('L')[0].split('_')[1])
lstmNumLayers = int(bestLstmName.split('L')[0].split('_')[1])

rnnEmbed, rnnProj, rnnOut, rnnModel = loadRnnModel(bestRnnPath, rnnNumLayers)
lstmEmbed, lstmProj, lstmOut, lstmModel = loadLstmModel(bestLstmPath, lstmNumLayers)

imageBleuScores = {}

for i, (imgName, caps) in enumerate(testCaps.items()):
    if i % 200 == 0:
        print(f"  {i}/{len(testCaps)}...", end='\r')
    
    if imgName not in imageFeatures:
        continue
    
    feat = imageFeatures[imgName]
    
    # Caption dari RNN
    rnnCaption = ' '.join(greedyDecode(rnnModel, rnnProj, rnnEmbed, rnnOut, feat, vocab, MAX_LEN))
    
    # Caption dari LSTM
    x = lstmProj.forward(feat)
    h = [np.zeros(cell.hiddenDim) for cell in lstmModel.cells]
    c = [np.zeros(cell.hiddenDim) for cell in lstmModel.cells]
    
    words = []
    token = vocab['<start>']
    for _ in range(MAX_LEN):
        x_emb = lstmEmbed.forward(token)
        h, c = lstmModel.forwardStep(x_emb, h, c)
        logits = lstmOut.forward(h[-1])
        token = int(np.argmax(logits))
        if token == vocab['<end>']:
            break
        words.append(id2word.get(token, '<unk>'))
    lstmCaption = ' '.join(words)
    
    # BLEU-4 untuk keduanya
    references = [cap.split() for cap in caps]
    rnnBleu = sentence_bleu(references, rnnCaption.split(), weights=(0.25, 0.25, 0.25, 0.25))
    lstmBleu = sentence_bleu(references, lstmCaption.split(), weights=(0.25, 0.25, 0.25, 0.25))
    
    # Rata-rata BLEU untuk gambar ini
    avgBleu = (rnnBleu + lstmBleu) / 2
    imageBleuScores[imgName] = {
        'bleu_avg': avgBleu,
        'bleu_rnn': rnnBleu,
        'bleu_lstm': lstmBleu,
        'caption_rnn': rnnCaption,
        'caption_lstm': lstmCaption,
        'reference': caps[0],
    }

print(f"\nSelesai hitung BLEU untuk {len(imageBleuScores)} gambar")


## Hitung BLEU-4 dan Pilih Sampel

Langkah 1: Buat caption dengan RNN terbaik dan LSTM terbaik untuk semua gambar uji.
Langkah 2: Hitung BLEU-4 per gambar sebagai ukuran kualitas:
- Skor tinggi (>0.35): isi gambar tertangkap dengan baik
- Skor sedang (0.15-0.35): sebagian benar, masih ada kesalahan
- Skor rendah (<0.15): caption jauh dari referensi

Langkah 3: Pilih sampel untuk analisis kualitatif:
- **Kuartil atas:** 3-4 gambar dengan BLEU tertinggi
- **Kuartil tengah:** 3-4 gambar dengan BLEU sedang
- **Kuartil bawah:** 3-4 gambar dengan BLEU terendah

Perbandingan RNN vs LSTM dari kuartil ini membantu melihat:
- Di mana LSTM konsisten lebih baik
- Pola kesalahan yang sering muncul
- Dampak cell state untuk dependensi panjang

## Visualisasi Kualitatif: RNN vs LSTM

**Grid 3x3 + 1 baris analitik:**

**Baris 1-3 (4 gambar per baris):**
- **Baris 1 (atas):** 4 gambar dengan BLEU tertinggi
- **Baris 2 (tengah):** 4 gambar dengan BLEU sedang
- **Baris 3 (bawah):** 4 gambar dengan BLEU terendah

Setiap sel menampilkan:
1. Gambar dari Flickr8k
2. Skor BLEU (RNN vs LSTM)
3. Caption hasil RNN dan LSTM
4. Caption referensi (ground truth)

**Baris 4 (analitik):**
- **Kiri:** Histogram selisih BLEU (LSTM - RNN). Positif = LSTM lebih baik
- **Tengah:** Scatter korelasi BLEU RNN vs LSTM
- **Kanan:** Ringkasan statistik (rata-rata, simpangan baku, persentase peningkatan)

**Inti yang diharapkan:**
1. **Kasus sukses:** kedua model cukup baik, LSTM sedikit lebih rapi
2. **Kasus gagal:** dua model sama-sama kesulitan, tetapi pola salahnya beda
3. **Keunggulan LSTM:** lebih stabil untuk caption panjang
4. **Sebaran:** titik banyak berada di atas diagonal (LSTM unggul)

In [ ]:
## Analisis Pola Error dan Perbedaan RNN vs LSTM

print("="*70)
print("ANALISIS ERROR RINCI")
print("="*70)

rnnBetter = 0
lstmBetter = 0
equal = 0

rnnOnlyErrors = []
lstmOnlyErrors = []

for img, score in imageBleuScores.items():
    if score['bleu_rnn'] > score['bleu_lstm'] + 0.01:
        rnnBetter += 1
        rnnOnlyErrors.append({
            'image': img,
            'diff': score['bleu_rnn'] - score['bleu_lstm'],
            'rnn_bleu': score['bleu_rnn'],
            'lstm_bleu': score['bleu_lstm'],
        })
    elif score['bleu_lstm'] > score['bleu_rnn'] + 0.01:
        lstmBetter += 1
        lstmOnlyErrors.append({
            'image': img,
            'diff': score['bleu_lstm'] - score['bleu_rnn'],
            'rnn_bleu': score['bleu_rnn'],
            'lstm_bleu': score['bleu_lstm'],
        })
    else:
        equal += 1

total = len(imageBleuScores)
print(f"\nDistribusi hasil:")
print(f"  LSTM lebih baik : {lstmBetter}/{total} ({lstmBetter/total*100:.1f}%)")
print(f"  RNN lebih baik  : {rnnBetter}/{total} ({rnnBetter/total*100:.1f}%)")
print(f"  Hampir sama     : {equal}/{total} ({equal/total*100:.1f}%)")

if rnnOnlyErrors:
    rnnOnlyErrors.sort(key=lambda x: x['diff'], reverse=True)
    print(f"\n{len(rnnOnlyErrors)} kasus RNN lebih unggul dari LSTM:")
    print(f"  Rata-rata selisih: {np.mean([e['diff'] for e in rnnOnlyErrors]):.4f}")
    print(f"  Selisih maksimum : {max(e['diff'] for e in rnnOnlyErrors):.4f}")
    print(f"\n  Top 3 keunggulan RNN:")
    for e in rnnOnlyErrors[:3]:
        print(f"    {e['image']}: RNN={e['rnn_bleu']:.3f} vs LSTM={e['lstm_bleu']:.3f} (selisih={e['diff']:.3f})")

if lstmOnlyErrors:
    lstmOnlyErrors.sort(key=lambda x: x['diff'], reverse=True)
    print(f"\n{len(lstmOnlyErrors)} kasus LSTM lebih unggul dari RNN:")
    print(f"  Rata-rata selisih: {np.mean([e['diff'] for e in lstmOnlyErrors]):.4f}")
    print(f"  Selisih maksimum : {max(e['diff'] for e in lstmOnlyErrors):.4f}")
    print(f"\n  Top 3 keunggulan LSTM:")
    for e in lstmOnlyErrors[:3]:
        print(f"    {e['image']}: LSTM={e['lstm_bleu']:.3f} vs RNN={e['rnn_bleu']:.3f} (selisih={e['diff']:.3f})")

print("\n" + "="*70)

In [ ]:
# Urutkan berdasarkan BLEU rata-rata dan pilih sampel kuartil
sortedImages = sorted(imageBleuScores.items(), key=lambda x: x[1]['bleu_avg'], reverse=True)
bleuValues = [score['bleu_avg'] for _, score in sortedImages]

print("Sebaran BLEU-4:")
print(f"  Minimum : {min(bleuValues):.4f}")
print(f"  Maksimum: {max(bleuValues):.4f}")
print(f"  Rata-rata: {np.mean(bleuValues):.4f}")
print(f"  Median  : {np.median(bleuValues):.4f}")

# Batas kuartil
q75 = np.percentile(bleuValues, 75)
q50 = np.percentile(bleuValues, 50)
q25 = np.percentile(bleuValues, 25)

print("\nBatas kuartil:")
print(f"  Q75 (75%) : {q75:.4f}")
print(f"  Q50 (50%) : {q50:.4f}")
print(f"  Q25 (25%) : {q25:.4f}")

# Ambil 4 sampel dari tiap kuartil
topImages = [img for img, score in sortedImages if score['bleu_avg'] >= q75][:4]
midImages = [img for img, score in sortedImages if q50 <= score['bleu_avg'] < q75][:4]
lowImages = [img for img, score in sortedImages if score['bleu_avg'] < q25][:4]

print("\nSampel terpilih:")
print(f"  Kuartil atas : {len(topImages)} gambar (BLEU >= {q75:.4f})")
print(f"  Kuartil tengah: {len(midImages)} gambar ({q50:.4f} <= BLEU < {q75:.4f})")
print(f"  Kuartil bawah : {len(lowImages)} gambar (BLEU < {q25:.4f})")

fig, axes = plt.subplots(4, 3, figsize=(18, 16))

def plotSample(ax, imgName, title):
    try:
        imgPath = FLICKR_DIR / f"{imgName}.jpg"
        if not imgPath.exists():
            ax.text(0.5, 0.5, f"Gambar tidak ditemukan:\n{imgName}",
                   ha='center', va='center', fontsize=10)
            ax.axis('off')
            return
        
        img = Image.open(imgPath)
        ax.imshow(img)
        ax.axis('off')
        
        score = imageBleuScores[imgName]
        caption_text = (
            f"{title}\n"
            f"BLEU: RNN={score['bleu_rnn']:.3f} | LSTM={score['bleu_lstm']:.3f}\n\n"
            f"RNN :  {score['caption_rnn']}\n\n"
            f"LSTM: {score['caption_lstm']}\n\n"
            f"REF :  {score['reference']}"
        )
        ax.set_title(caption_text, fontsize=8, loc='left', wrap=True,
                    fontfamily='monospace', verticalalignment='top')
        
    except Exception as e:
        ax.text(0.5, 0.5, f"Galat: {e}", ha='center', va='center')
        ax.axis('off')

# Plot kuartil atas
for i, imgName in enumerate(topImages):
    plotSample(axes[0, i], imgName, "ATAS (BLEU tinggi)")

# Plot kuartil tengah
for i, imgName in enumerate(midImages):
    plotSample(axes[1, i], imgName, "TENGAH (BLEU sedang)")

# Plot kuartil bawah
for i, imgName in enumerate(lowImages):
    plotSample(axes[2, i], imgName, "BAWAH (BLEU rendah)")

# Plot statistik gabungan
ax = axes[3, 0]
diff_bleus = [imageBleuScores[img]['bleu_lstm'] - imageBleuScores[img]['bleu_rnn']
              for img in imageBleuScores]
ax.hist(diff_bleus, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Performa setara')
ax.axvline(x=np.mean(diff_bleus), color='green', linestyle='-', linewidth=2,
          label=f'Rata-rata: {np.mean(diff_bleus):.4f}')
ax.set_xlabel('Selisih BLEU-4 (LSTM - RNN)')
ax.set_ylabel('Frekuensi')
ax.set_title('Distribusi Performa LSTM vs RNN')
ax.legend()
ax.grid(True, alpha=0.3)

# Scatter plot
ax = axes[3, 1]
rnn_bleus = [imageBleuScores[img]['bleu_rnn'] for img in imageBleuScores]
lstm_bleus = [imageBleuScores[img]['bleu_lstm'] for img in imageBleuScores]
ax.scatter(rnn_bleus, lstm_bleus, alpha=0.3, s=10)
ax.plot([0, max(rnn_bleus)], [0, max(rnn_bleus)], 'r--', label='Performa setara')
ax.set_xlabel('BLEU-4 RNN')
ax.set_ylabel('BLEU-4 LSTM')
ax.set_title('Korelasi RNN vs LSTM')
ax.legend()
ax.grid(True, alpha=0.3)

# Ringkasan statistik
ax = axes[3, 2]
ax.axis('off')
summary_text = (
    f"Ringkasan Statistik\n"
    f"{'='*30}\n"
    f"Jumlah gambar uji: {len(imageBleuScores)}\n\n"
    f"RNN:\n"
    f"  Rata-rata BLEU: {np.mean(rnn_bleus):.4f}\n"
    f"  Simpangan baku: {np.std(rnn_bleus):.4f}\n\n"
    f"LSTM:\n"
    f"  Rata-rata BLEU: {np.mean(lstm_bleus):.4f}\n"
    f"  Simpangan baku: {np.std(lstm_bleus):.4f}\n\n"
    f"LSTM lebih baik: {sum(1 for l, r in zip(lstm_bleus, rnn_bleus) if l > r)}/{len(imageBleuScores)} kasus\n"
    f"Rata-rata peningkatan: {np.mean(diff_bleus):.4f}"
 )
ax.text(0.1, 0.9, summary_text, fontsize=10, verticalalignment='top',
       fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle(f"Analisis Kualitatif: RNN terbaik vs LSTM terbaik\n{bestRnnName} vs {bestLstmName}",
            fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "src/wajib/weights/qualitative_analysis.png", dpi=100, bbox_inches='tight')
plt.show()



## Teori Singkat: Kenapa LSTM Lebih Baik dari RNN

### Masalah Vanishing Gradient
**RNN:** $h_t = tanh(x_t W_x + h_{t-1} W_h + b)$
- Gradien lewat satu fungsi tanh tiap langkah
- Semakin jauh langkahnya, gradien makin kecil
- Model mudah lupa informasi jauh (lebih dari 20-30 langkah)

**LSTM:**
- Ada cell state yang mengalir lebih stabil
- Forget gate mengatur seberapa banyak informasi disimpan
- Dependensi jauh (puluhan langkah) lebih terjaga

### Konteks Image Captioning
- Rata-rata caption Flickr8k sekitar 11-12 kata
- Informasi penting (objek, warna, posisi) harus tersimpan
- LSTM cenderung lebih stabil untuk caption yang panjang

### Ringkasnya
- **Secara empiris:** LSTM sering memberi BLEU-4 lebih tinggi
- **Secara teori:** gate pada LSTM membantu menjaga memori
- **Trade-off:** LSTM sedikit lebih lambat, tapi kualitas biasanya naik

### Catatan Praktis
- Kualitas data dan CNN encoder sangat berpengaruh
- Optimizer dan batch size perlu disetel
- Early stopping bisa mencegah overfitting